In [6]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time

from CFD.Riemann_Solvers_and_Numerical_Methods_for_Fluid_Dynamics_by_TORO.chapter13_14 import muscl_reconstruction, reflective_bc
from CFD.Riemann_Solvers_and_Numerical_Methods_for_Fluid_Dynamics_by_TORO.chapter16 import strang_update
from CFD.Riemann_Solvers_and_Numerical_Methods_for_Fluid_Dynamics_by_TORO.chapter10 import HLLC_Riemann_Solver
from CFD.Riemann_Solvers_and_Numerical_Methods_for_Fluid_Dynamics_by_TORO.chapter6 import exact_riemann_flux
from simulation.simulator import Simulator
from CFD import LinesOnVolume
from CFD import Dye
from CFD import create_random_circle_points
from CFD import create_random_sphere_points
from CFD import create_explosion_initial_condition

# GPU 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"CUDA device: {torch.cuda.get_device_name(0)}")

Using device: cuda
CUDA device: NVIDIA GeForce RTX 5080


In [ ]:
# reference unit
rho0 = 1.2
p0   = 1e5
u0 = (p0 / rho0)**0.5

#Constant parameters
#z, y, x
#for 2dim simulateion set z_domain to 1
tmp = 320
RESOLUTION = tmp, tmp, tmp

X_DOMAIN = [0, 1] # 1 m
Y_DOMAIN = [0, 1] # 1 m
Z_DOMAIN = [0, 1] # 1 m
T_DOMAIN = [0, 0.9]
DX = (X_DOMAIN[1] - X_DOMAIN[0]) / RESOLUTION[2]
DY = (Y_DOMAIN[1] - Y_DOMAIN[0]) / RESOLUTION[1]
DZ = (Z_DOMAIN[1] - Z_DOMAIN[0]) / RESOLUTION[0]

CFL_COEFFICIENT = 0.6
GAMMA = 1.4
TOL = 1e-6

# Initial Conditions
rho_inner_phy = 3 #(kg/m³)
p_inner_phy = 1e7 #(pa)
rho_outer_phy = 1.225 #(kg/m³)
p_outer_phy = 101325 #(pa)

# convert to reference unit
rho_inner = rho_inner_phy / rho0
p_inner = p_inner_phy / p0
rho_outer = rho_outer_phy / rho0
p_outer = p_outer_phy / p0

sigma = 0.01

In [8]:
def mask_sphere_solid_mask(solid_cell):
    """
    solid_cell: (Nz, Ny, Nx)
    중심 기준 가장 큰 원 내부=0, 외부=1 로 설정
    """

    Nz, Ny, Nx = solid_cell.shape
    device = solid_cell.device

    # 좌표 생성
    y = torch.arange(Ny, device=device)
    x = torch.arange(Nx, device=device)
    z = torch.arange(Nz, device=device)

    yy, xx, zz = torch.meshgrid(y, x, z, indexing='ij')

    # 중심 좌표
    cy = (Ny - 1) / 2
    cx = (Nx - 1) / 2
    cz = (Nz - 1) / 2
    # 반지름 (inscribed circle)
    r = min(Nx, Ny, Nz) / 2 - 1

    # 거리 계산
    dist2 = (yy - cy)**2 + (xx - cx)**2 + (zz - cz)**2

    # 원 내부: 0, 외부: 1
    circle_mask = dist2 > r**2   # True = 외부

    return circle_mask

def make_circular_solid_mask(solid_cell):
    """
    solid_cell: (Nz, Ny, Nx)
    중심 기준 가장 큰 원 내부=0, 외부=1 로 설정
    """

    Nz, Ny, Nx = solid_cell.shape
    device = solid_cell.device

    # 좌표 생성
    y = torch.arange(Ny, device=device)
    x = torch.arange(Nx, device=device)

    yy, xx = torch.meshgrid(y, x, indexing='ij')

    # 중심 좌표
    cy = (Ny - 1) / 2
    cx = (Nx - 1) / 2

    # 반지름 (inscribed circle)
    r = min(Nx, Ny) / 2 - 1

    # 거리 계산
    dist2 = (yy - cy)**2 + (xx - cx)**2

    # 원 내부: 0, 외부: 1
    circle_mask = dist2 > r**2   # True = 외부

    return solid_cell

In [9]:

CELL = create_explosion_initial_condition(
    nx = RESOLUTION[2],
    ny = RESOLUTION[1],
    nz = RESOLUTION[0],
    x_domain = X_DOMAIN,
    y_domain = Y_DOMAIN,
    z_domain = Z_DOMAIN,
    rho_inner = rho_inner,
    p_inner = p_inner,
    rho_outer = rho_outer,
    p_outer = p_outer,
    sigma = sigma,
    noise = 0.2,
    r_k0_list = [1/11, 1/7, 1/5],
    weight_list = [1.0, 0.7, 0.3],
    device = device
)


source = create_explosion_initial_condition(
    nx = RESOLUTION[2],
    ny = RESOLUTION[1],
    nz = RESOLUTION[0],
    x_domain = X_DOMAIN,
    y_domain = Y_DOMAIN,
    z_domain = Z_DOMAIN,
    rho_inner = rho_inner,
    p_inner = p_inner,
    rho_outer = rho_outer,
    p_outer = p_outer,
    sigma = sigma,
    noise = 0.0,
    device = device
)

#for 2dim simulateion use create_random_circle_points instead of create_random_sphere_points.
polylines = create_random_sphere_points(0.03, 1000, 100, X_DOMAIN, Y_DOMAIN, Z_DOMAIN, device)
lov = LinesOnVolume(RESOLUTION[2] * 5, RESOLUTION[1] * 5, polylines, max_length=1)

dye = Dye(torch.ones_like(source[..., 0]))

solid_cell = torch.zeros_like(CELL[..., 0])
solid_cell = mask_sphere_solid_mask(solid_cell)


sim = Simulator(
    CELL,
    dx=DX,
    dy=DY,
    dz=DZ,
    riemann_solver=HLLC_Riemann_Solver,
    reconstruction_method=muscl_reconstruction,
    boundary_function=reflective_bc,
    update_method=strang_update,
    solid_cell = solid_cell.bool(),
    dimension=3,
    visualizers=[]
)

In [ ]:
import shutil
import os

folder = "/TMP_MOUNT/cells"

shutil.rmtree(folder, ignore_errors=True)
os.makedirs(folder, exist_ok=True)

fig, ax = plt.subplots(1, sim.get_images_num(), figsize=(30, 15), dpi=150)

# visualization settings
plt.tight_layout()
for a in ax:
    a.axis('off')


images = sim.get_images()
canvas = []
for i in range(sim.get_images_num()):
    tmp = ax[i].imshow(images[i], origin='lower', cmap='viridis')
    canvas.append(tmp)
plt.show()

t = T_DOMAIN[0]
counter = 0
while t < T_DOMAIN[1]:
    for inner in range(3):
        counter += 1
        dt = sim.update()
        t += dt

        #update the center rho, p to maintain the explosion  * torch.exp(-t)
        sim.cell[..., 0] = torch.max(sim.cell[..., 0], source[..., 0])
        sim.cell[..., 4] = torch.max(sim.cell[..., 4], source[..., 4])

        #dye.dye_field = torch.max(dye.dye_field, source[..., 0])

    torch.save(sim.cell.half(), f"/TMP_MOUNT/cells/{t}.pt")

    '''
    images = sim.get_images()
    for i in range(sim.get_images_num()):
        canvas[i].set_data(images[i])
        canvas[i].set_clim(images[i].min(), images[i].max())

    clear_output(wait=True) 
    display(fig)


    torch.cuda.synchronize()

    time.sleep(0.001)
    '''
    clear_output(wait=True) 
    print(f"t = {t:.3f}")


t = 0.420
